# MS/MS Data Preprocessing

## Objectives

1. Create a reproducible structure-level train/validation split.
2. Convert MS/MS spectra into fixed-length CNN inputs.
3. Generate Morgan fingerprint labels from SMILES.
4. Prepare PyTorch Dataset and DataLoader.

## Data Leakage Prevention

We split the dataset by `inchikey14` rather than by
individual spectra.

All spectra belonging to the same molecular structure
must remain in the same split.

## 1. Data Splitting

In [ ]:
import kagglehub

kagglehub.login()

In [ ]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split

COMPETITION = "enveda-CASMI26-molecule-id-mass-spectra"

data_dir = Path(
    kagglehub.competition_download(COMPETITION)
)

TRAIN_FILE = next(data_dir.rglob("train.parquet"))

pf = pq.ParquetFile(TRAIN_FILE)

unique_keys = set()
missing_keys = 0

for batch in pf.iter_batches(
    batch_size=100000,
    columns=["inchikey14"]
):
    keys = batch.column(0).to_pylist()

    for key in keys:
        if isinstance(key, str) and key.strip():
            unique_keys.add(key)
        else:
            missing_keys += 1

print("Total spectra:", pf.metadata.num_rows)
print("Unique structures:", len(unique_keys))
print("Missing structure IDs:", missing_keys)

In [ ]:
structure_ids = sorted(unique_keys)

train_ids, val_ids = train_test_split(
    structure_ids,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

train_ids = set(train_ids)
val_ids = set(val_ids)

print("Train structures:", len(train_ids))
print("Validation structures:", len(val_ids))

# Verify no structure overlap
overlap = train_ids.intersection(val_ids)

print("Overlapping structures:", len(overlap))

assert len(overlap) == 0
assert len(train_ids) + len(val_ids) == len(unique_keys)

print("Structure-level split passed!")

In [ ]:
import json

split_info = {
    "random_seed": 42,
    "split_method": "structure_level",
    "train_structures": sorted(train_ids),
    "val_structures": sorted(val_ids)
}

with open("structure_split.json", "w") as f:
    json.dump(split_info, f)

print("Saved structure_split.json")

## 2. Molecular Fingerprint Label Generation

We use RDKit to convert molecular structures into
1024-bit Morgan fingerprints.

### Configuration
- Fingerprint type: Morgan
- Radius: 2
- Number of bits: 1024
- Include chirality: False

### Purpose
The CNN models will learn to predict molecular
fingerprints from MS/MS spectral features.

This allows us to compare predicted fingerprints
with candidate molecular structures during ranking.

We use the same fingerprint configuration for
both CNN architectures.

In [ ]:
!pip -q install rdkit

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator


In [ ]:
smiles_by_key = {}
conflicting_smiles = 0
missing_smiles = 0

for batch in pf.iter_batches(
    batch_size=50000,
    columns=["inchikey14", "normalized_smiles"]
):
    keys = batch.column(0).to_pylist()
    smiles_list = batch.column(1).to_pylist()

    for key, smiles in zip(keys, smiles_list, strict=False):

        if not isinstance(smiles, str) or not smiles.strip():
            missing_smiles += 1
            continue

        if key not in smiles_by_key:
            smiles_by_key[key] = smiles

        elif smiles_by_key[key] != smiles:
            conflicting_smiles += 1

print("Unique structures with SMILES:", len(smiles_by_key))
print("Rows with missing SMILES:", missing_smiles)
print("Rows with differing SMILES for the same key:", conflicting_smiles)

print("\nExample:")
for key, smiles in list(smiles_by_key.items())[:3]:
    print(key, ":", smiles)

In [ ]:
FP_SIZE = 1024
FP_RADIUS = 2

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS,
    fpSize=FP_SIZE,
    includeChirality=False
)

def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fp = morgan_generator.GetFingerprint(mol)

    arr = np.zeros(FP_SIZE, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr

In [ ]:
sample_keys = list(smiles_by_key.keys())[:100]

fingerprints = []
failed_keys = []

for key in sample_keys:
    smiles = smiles_by_key[key]

    fp = smiles_to_fingerprint(smiles)

    if fp is None:
        failed_keys.append(key)
    else:
        fingerprints.append(fp)

X_fp = np.stack(fingerprints) if fingerprints else None

print("Structures tested:", len(sample_keys))
print("Successful fingerprints:", len(fingerprints))
print("Failed structures:", len(failed_keys))

if X_fp is not None:
    print("Fingerprint matrix shape:", X_fp.shape)
    print("Data type:", X_fp.dtype)

    print("\nFirst fingerprint (first 50 bits):")
    print(X_fp[0][:50])

    print("\nNumber of active bits:")
    print(int(X_fp[0].sum()))

## 3. SMILES Consistency Audit

During label mapping, 33,631 rows had SMILES strings
that differed from the first SMILES associated with
the same InChIKey14.

This does not necessarily indicate incorrect labels.

We will examine:
1. How many unique InChIKey14 groups contain multiple SMILES.
2. Whether these differences affect Morgan fingerprints.
3. Whether fingerprint labels can be generated consistently.

For model training, the original row-level SMILES
labels will be preserved unless a consistent
normalization strategy is established.

In [ ]:
from collections import Counter, defaultdict

alternate_smiles = defaultdict(set)

for batch in pf.iter_batches(
    batch_size=50000,
    columns=["inchikey14", "normalized_smiles"]
):
    keys = batch.column(0).to_pylist()
    smiles_list = batch.column(1).to_pylist()

    for key, smiles in zip(keys, smiles_list, strict=False):
        if key in smiles_by_key and smiles != smiles_by_key[key]:
            alternate_smiles[key].add(smiles)

print("Unique InChIKey14 groups with multiple SMILES:",
      len(alternate_smiles))

print("\nExamples:")

for key in list(alternate_smiles.keys())[:5]:
    print("\nInChIKey14:", key)
    print("Original:", smiles_by_key[key])

    for s in list(alternate_smiles[key])[:3]:
        print("Alternative:", s)

In [ ]:
fp_conflict_keys = []
fp_same_keys = []
invalid_smiles_keys = []

for key, alternatives in alternate_smiles.items():

    all_smiles = [smiles_by_key[key], *alternatives]

    fp_values = []
    invalid = False

    for smiles in all_smiles:
        fp = smiles_to_fingerprint(smiles)

        if fp is None:
            invalid = True
            break

        fp_values.append(fp.tobytes())

    if invalid:
        invalid_smiles_keys.append(key)
    elif len(set(fp_values)) > 1:
        fp_conflict_keys.append(key)
    else:
        fp_same_keys.append(key)

print("Groups with identical fingerprints:",
      len(fp_same_keys))

print("Groups with different fingerprints:",
      len(fp_conflict_keys))

print("Groups with invalid SMILES:",
      len(invalid_smiles_keys))

assert (
    len(fp_same_keys)
    + len(fp_conflict_keys)
    + len(invalid_smiles_keys)
    == len(alternate_smiles)
)

## 4. Tautomer Standardization

The label consistency audit identified 1,612 InChIKey14
groups containing multiple SMILES representations.

Among these groups, 1,611 produced different Morgan
fingerprints before standardization.

Since the competition evaluates molecular structures
after tautomer canonicalization, we will investigate
whether RDKit tautomer standardization improves
fingerprint label consistency.

We will compare fingerprints before and after
standardization without removing any training records.

In [ ]:
from functools import lru_cache

import numpy as np
from rdkit.Chem.MolStandardize import rdMolStandardize

tautomer_enumerator = rdMolStandardize.TautomerEnumerator()

@lru_cache(maxsize=10000)
def canonical_smiles_to_fingerprint(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    # Stereochemistry is not part of our target
    Chem.RemoveStereochemistry(mol)

    # Canonicalize tautomer
    canonical_mol = tautomer_enumerator.Canonicalize(mol)

    canonical_smiles = Chem.MolToSmiles(
        canonical_mol,
        canonical=True
    )

    # Generate Morgan fingerprint
    fp = morgan_generator.GetFingerprint(canonical_mol)

    arr = np.zeros(FP_SIZE, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    return canonical_smiles, arr.tobytes()

In [ ]:
rng = np.random.default_rng(42)

all_conflict_keys = sorted(alternate_smiles.keys())

selected_keys = rng.choice(
    all_conflict_keys,
    size=min(100, len(all_conflict_keys)),
    replace=False
)

identical_after = 0
different_after = 0
failed_groups = 0

for key in selected_keys:

    smiles_variants = [
        smiles_by_key[key],
        *sorted(alternate_smiles[key])
    ]

    results = [
        canonical_smiles_to_fingerprint(s)
        for s in smiles_variants
    ]

    if any(r is None for r in results):
        failed_groups += 1
        continue

    fingerprint_values = [r[1] for r in results]

    if len(set(fingerprint_values)) == 1:
        identical_after += 1
    else:
        different_after += 1

print("Groups tested:", len(selected_keys))
print("Identical fingerprints after standardization:",
      identical_after)
print("Different fingerprints after standardization:",
      different_after)
print("Failed groups:", failed_groups)

In [ ]:
from rdkit import rdBase
from rdkit.Chem.MolStandardize import rdMolStandardize

print("Local RDKit version:", rdBase.rdkitVersion)
print("Competition scorer RDKit version: 2026.03.3")

# Reuse existing tautomer enumerator
def get_scoring_key(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    try:
        canonical_mol = tautomer_enumerator.Canonicalize(mol)
        return Chem.MolToInchiKey(canonical_mol)[:14]
    except Exception:
        return None


# Find the 14 unresolved groups from our sample
problem_keys = []

for key in selected_keys:
    variants = [
        smiles_by_key[key],
        *sorted(alternate_smiles[key])
    ]

    fingerprints = [
        canonical_smiles_to_fingerprint(s)
        for s in variants
    ]

    if (any(fp is None for fp in fingerprints)
        or len({fp[1] for fp in fingerprints}) > 1):
        problem_keys.append(key)

print("Unresolved groups:", len(problem_keys))


# Compare fingerprint and scoring-key consistency
audit_results = []

for key in problem_keys:
    variants = [
        smiles_by_key[key],
        *sorted(alternate_smiles[key])
    ]

    fp_results = [
        canonical_smiles_to_fingerprint(s)
        for s in variants
    ]

    scoring_keys = [
        get_scoring_key(s)
        for s in variants
    ]

    audit_results.append({
        "original_key": key,
        "n_smiles": len(variants),
        "n_fingerprints": len({
            r[1] for r in fp_results if r is not None
        }),
        "n_scoring_keys": len(set(scoring_keys)),
        "has_scoring_error": any(
            k is None for k in scoring_keys
        )
    })

df_conflict_audit = pd.DataFrame(audit_results)

display(df_conflict_audit)

print("\nScoring-key consistency:")
print(df_conflict_audit["n_scoring_keys"].value_counts())

In [ ]:
conflict_keys = set(alternate_smiles.keys())

total_rows = 0
affected_rows = 0

for batch in pf.iter_batches(
    batch_size=100000,
    columns=["inchikey14"]
):
    keys = batch.column(0).to_pylist()

    total_rows += len(keys)
    affected_rows += sum(
        key in conflict_keys for key in keys
    )

print("Total spectra:", total_rows)
print("Spectra in conflicting groups:", affected_rows)

print(
    "Affected spectra percentage:",
    f"{affected_rows / total_rows * 100:.4f}%"
)

print(
    "Affected structure groups:",
    f"{len(conflict_keys) / len(unique_keys) * 100:.4f}%"
)

## 6. Label Consistency and RDKit Version Alignment

The SMILES consistency audit identified 1,612 groups
with multiple representations, covering 3.6754% of
training spectra.

Among 100 sampled conflicting groups, 14 retained
different Morgan fingerprints after tautomer
canonicalization. Twelve shared one scoring key,
while two produced different scoring keys.

The local RDKit version was 2026.09.1, while the
competition specifies RDKit 2026.03.3.

We will align the RDKit version, repeat the audit,
and verify the validation split using canonicalized
structure keys before model training.

In [ ]:
import json
import os
import shutil
from pathlib import Path


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return current

# Set these environment variables to override the local defaults.
REPO_ROOT = Path(
    os.environ.get("CASMI_REPO_ROOT", find_repo_root())
).expanduser().resolve()
SAVE_DIR = Path(
    os.environ.get(
        "CASMI_MEMBER2_DIR",
        REPO_ROOT / "outputs" / "member2"
    )
).expanduser().resolve()
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Member 2 working directory:", SAVE_DIR)

# Save all conflicting SMILES groups
all_conflicts = {
    str(key): [
        smiles_by_key[key],
        *sorted(alternate_smiles[key])
    ]
    for key in alternate_smiles
}

audit_data = {
    "all_conflicts": all_conflicts,
    "unresolved_keys": [str(k) for k in problem_keys]
}

with open(SAVE_DIR / "smiles_audit.json", "w") as f:
    json.dump(audit_data, f)

# Preserve the original split for reference
with open(SAVE_DIR / "structure_split_original.json", "w") as f:
    json.dump({
        "train": sorted(train_ids),
        "validation": sorted(val_ids),
        "seed": 42
    }, f)

print("Saved to:", SAVE_DIR)
print("Conflicting groups:", len(all_conflicts))
print("Unresolved sample groups:", len(problem_keys))

In [ ]:
!pip install -q "rdkit==2026.3.3"

In [ ]:
import json

import pandas as pd
from rdkit import rdBase
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize

print("RDKit version:", rdBase.rdkitVersion)

assert rdBase.rdkitVersion == "2026.03.3"

with open(SAVE_DIR / "smiles_audit.json") as f:
    audit_data = json.load(f)

te = rdMolStandardize.TautomerEnumerator()

fp_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=1024,
    includeChirality=False
)

def evaluate_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return None

        Chem.RemoveStereochemistry(mol)
        canonical_mol = te.Canonicalize(mol)

        scoring_key = Chem.MolToInchiKey(
            canonical_mol
        )[:14]

        fp = fp_gen.GetFingerprint(canonical_mol)

        return scoring_key, fp.ToBitString()

    except Exception:
        return None


results = []

for key in audit_data["unresolved_keys"]:
    variants = audit_data["all_conflicts"][key]

    outputs = [
        evaluate_smiles(s)
        for s in variants
    ]

    valid = [x for x in outputs if x is not None]

    results.append({
        "original_key": key,
        "n_smiles": len(variants),
        "n_scoring_keys": len(set(x[0] for x in valid)),
        "n_fingerprints": len(set(x[1] for x in valid)),
        "n_errors": len(outputs) - len(valid)
    })

df_recheck = pd.DataFrame(results)

display(df_recheck)

print("\nScoring-key consistency:")
print(df_recheck["n_scoring_keys"].value_counts())

print("\nFingerprint consistency:")
print(df_recheck["n_fingerprints"].value_counts())

print("\nTotal errors:", df_recheck["n_errors"].sum())

## 7. Canonical Scoring Key Audit

The local RDKit version now matches the competition's
specified version (2026.03.3).

We will compute tautomer-canonicalized InChIKey14 values
for all distinct molecular representations.

This audit aims to:

1. Identify structures that share the same scoring key.
2. Detect scoring-equivalent structures with different raw keys.
3. Identify unresolved or invalid representations.
4. Prepare a leakage-safe train/validation split.

Original SMILES and fingerprints will be preserved.

In [ ]:
import json
from pathlib import Path

import kagglehub
import pyarrow.parquet as pq

COMPETITION = "enveda-CASMI26-molecule-id-mass-spectra"

data_dir = Path(
    kagglehub.competition_download(COMPETITION)
)

TRAIN_FILE = next(data_dir.rglob("train.parquet"))
pf = pq.ParquetFile(TRAIN_FILE)

with open(SAVE_DIR / "smiles_audit.json") as f:
    audit_data = json.load(f)

# One representative SMILES for each original key
first_smiles = {}

for batch in pf.iter_batches(
    batch_size=100000,
    columns=["inchikey14", "normalized_smiles"]
):
    keys = batch.column(0).to_pylist()
    smiles = batch.column(1).to_pylist()

    for key, smi in zip(keys, smiles, strict=False):
        if key not in first_smiles:
            first_smiles[key] = smi

print("Unique original keys:", len(first_smiles))
print("Groups with multiple SMILES:",
      len(audit_data["all_conflicts"]))

assert len(first_smiles) == 275810

In [ ]:
import json
import os

from rdkit import rdBase
from rdkit.Chem.MolStandardize import rdMolStandardize
from tqdm.auto import tqdm

assert rdBase.rdkitVersion == "2026.03.3"

te = rdMolStandardize.TautomerEnumerator()

def compute_scoring_key(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        Chem.RemoveStereochemistry(mol)
        mol = te.Canonicalize(mol)

        return Chem.MolToInchiKey(mol)[:14]

    except Exception:
        return None


keys = sorted(first_smiles)
conflicts = audit_data["all_conflicts"]

OUTPUT_DIR = SAVE_DIR / "scoring_key_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 2500

for start in tqdm(range(0, len(keys), CHUNK_SIZE)):

    output_file = OUTPUT_DIR / f"chunk_{start:06d}.json"

    # Resume previously completed chunks
    if output_file.exists():
        continue

    records = []

    for key in keys[start:start + CHUNK_SIZE]:

        variants = conflicts.get(
            key, [first_smiles[key]]
        )

        scoring_keys = set()
        n_errors = 0

        for smiles in variants:

            sk = compute_scoring_key(smiles)

            if sk is None:
                n_errors += 1
            else:
                scoring_keys.add(sk)

        records.append({
            "original_key": key,
            "scoring_keys": sorted(scoring_keys),
            "n_variants": len(variants),
            "n_errors": n_errors
        })

    # Atomic save: avoid keeping incomplete chunks
    temp_file = output_file.with_suffix(".tmp")

    with open(temp_file, "w") as f:
        json.dump(records, f)

    os.replace(temp_file, output_file)

print("Scoring key audit chunks saved.")

## 8. Full Scoring Key Audit Results

We completed the scoring-key audit using RDKit 2026.03.3.

This section summarizes:
- Total audited structures
- Missing or failed scoring keys
- Structures mapping to multiple scoring keys
- Potential scoring-key overlaps between original structures

In [ ]:
import json
from collections import defaultdict

files = sorted(OUTPUT_DIR.glob("chunk_*.json"))

records = []

for file in files:
    with open(file) as f:
        records.extend(json.load(f))

print("Completed chunks:", len(files))
print("Total audited structures:", len(records))
print("Unique original keys:",
      len(set(r["original_key"] for r in records)))

print("\nStructures with errors:",
      sum(r["n_errors"] > 0 for r in records))

print("Structures with multiple scoring keys:",
      sum(len(r["scoring_keys"]) > 1 for r in records))

print("Structures without scoring keys:",
      sum(len(r["scoring_keys"]) == 0 for r in records))

assert len(files) == 111
assert len(records) == 275810
assert len(set(r["original_key"] for r in records)) == 275810

print("\nFull audit completed successfully!")

In [ ]:
train_scoring_keys = set()
val_scoring_keys = set()

# Track scoring keys that map to multiple original keys
scoring_to_original = defaultdict(set)

for r in records:
    original_key = r["original_key"]

    for sk in r["scoring_keys"]:
        scoring_to_original[sk].add(original_key)

        if original_key in train_ids:
            train_scoring_keys.add(sk)

        if original_key in val_ids:
            val_scoring_keys.add(sk)

overlap = train_scoring_keys & val_scoring_keys

shared_scoring_keys = {
    sk: keys
    for sk, keys in scoring_to_original.items()
    if len(keys) > 1
}

print("Train scoring keys:", len(train_scoring_keys))
print("Validation scoring keys:", len(val_scoring_keys))

print("\nOverlapping scoring keys:", len(overlap))

print("Scoring keys shared by multiple original keys:",
      len(shared_scoring_keys))

print("\nExample overlapping scoring keys:")
for sk in sorted(overlap)[:10]:
    print(sk, "->", sorted(scoring_to_original[sk]))

## 9. Leakage-Safe Train/Validation Split

The full scoring-key audit identified:

- 275,810 original structure groups
- 287 overlapping scoring keys in the initial split
- 1,541 scoring keys shared by multiple original keys
- 11 original keys associated with multiple scoring keys

The initial split based on raw InChIKey14 was therefore
not fully leakage-safe.

We use a connected-component approach to group all
original structures linked by the same canonical scoring key.

Each connected component is assigned entirely to either
training or validation.

This prevents scoring-equivalent structures identified
by our audit from appearing in both sets.

In [ ]:
from collections import defaultdict

from sklearn.model_selection import train_test_split

# Each original key begins as its own component
parent = {
    r["original_key"]: r["original_key"]
    for r in records
}

size = {key: 1 for key in parent}


def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]


def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a == root_b:
        return

    if size[root_a] < size[root_b]:
        root_a, root_b = root_b, root_a

    parent[root_b] = root_a
    size[root_a] += size[root_b]


# Connect original structures sharing a scoring key
scoring_owner = {}

for r in records:
    original_key = r["original_key"]

    for sk in r["scoring_keys"]:
        if sk in scoring_owner:
            union(original_key, scoring_owner[sk])
        else:
            scoring_owner[sk] = original_key


# Build connected components
components = defaultdict(list)

for key in parent:
    components[find(key)].append(key)

component_ids = sorted(components.keys())

print("Original structure groups:", len(parent))
print("Connected components:", len(components))
print(
    "Components containing multiple original keys:",
    sum(len(v) > 1 for v in components.values())
)

print(
    "Largest component size:",
    max(len(v) for v in components.values())
)

In [ ]:
train_components, val_components = train_test_split(
    component_ids,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

train_ids = {
    key
    for comp in train_components
    for key in components[comp]
}

val_ids = {
    key
    for comp in val_components
    for key in components[comp]
}

print("Training structures:", len(train_ids))
print("Validation structures:", len(val_ids))

assert train_ids.isdisjoint(val_ids)
assert len(train_ids) + len(val_ids) == len(parent)

print("Original structure overlap: 0")

In [ ]:
train_scoring_keys = set()
val_scoring_keys = set()

for r in records:
    original_key = r["original_key"]

    if original_key in train_ids:
        train_scoring_keys.update(r["scoring_keys"])

    elif original_key in val_ids:
        val_scoring_keys.update(r["scoring_keys"])


scoring_overlap = train_scoring_keys & val_scoring_keys

print("Training scoring keys:", len(train_scoring_keys))
print("Validation scoring keys:", len(val_scoring_keys))

print("Overlapping scoring keys:", len(scoring_overlap))

assert len(scoring_overlap) == 0

print("Leakage-safe split validation passed!")

In [ ]:
import json

final_split = {
    "method": "scoring_key_connected_components",
    "random_seed": 42,
    "rdkit_version": "2026.03.3",
    "train_structures": sorted(train_ids),
    "val_structures": sorted(val_ids)
}

output_path = (
    SAVE_DIR / "structure_split_scoring_safe.json"
)

with open(output_path, "w") as f:
    json.dump(final_split, f)

print("Saved:", output_path)

## 10. CNN Training Data Preparation

We use the finalized scoring-key-connected-component
split to prepare training and validation data.

### Input Features
- Fixed-length binned MS/MS spectra
- m/z range: [0, 2200)
- Bin width: 1.0 Da
- Square-root intensity transformation

### Prediction Targets
- 1024-bit Morgan fingerprints
- Radius: 2
- Chirality: disabled
- Generated from row-level SMILES

### Initial Label Policy
For the first training experiment, we exclude
known ambiguous original structure groups from
the training data.

The validation set remains unchanged.

This is a preliminary modeling decision and will
be evaluated through later ablation experiments.

In [ ]:
import json

FINAL_SPLIT_PATH = (
    SAVE_DIR / "structure_split_scoring_safe.json"
)

with open(FINAL_SPLIT_PATH) as f:
    final_split = json.load(f)

train_ids = set(final_split["train_structures"])
val_ids = set(final_split["val_structures"])

print("Train structures:", len(train_ids))
print("Validation structures:", len(val_ids))

assert train_ids.isdisjoint(val_ids)
assert len(train_ids) + len(val_ids) == 275810

# Known ambiguous original structure groups
ambiguous_keys = set(
    audit_data["all_conflicts"].keys()
)

print("Known ambiguous groups:", len(ambiguous_keys))

print("Final split loaded successfully!")

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa

rng = np.random.default_rng(42)

columns = [
    "inchikey14",
    "normalized_smiles",
    "ms2_mzs",
    "ms2_normalized_intensities",
    "precursor_mz",
    "adduct",
    "ionization_mode",
    "ingest_lib"
]

sample_parts = []

for batch in pf.iter_batches(
    batch_size=4096,
    columns=columns
):
    # Approximately 0.8% random sample
    indices = np.flatnonzero(
        rng.random(batch.num_rows) < 0.008
    )

    if len(indices) > 0:
        selected = batch.take(pa.array(indices))
        sample_parts.append(selected.to_pandas())

df_model = pd.concat(
    sample_parts,
    ignore_index=True
)

# Training: use finalized split and exclude
# known ambiguous groups for the first experiment
df_train = df_model[
    df_model["inchikey14"].isin(train_ids)
    & ~df_model["inchikey14"].isin(ambiguous_keys)
].reset_index(drop=True)

# Validation: retain all valid structure groups
df_val = df_model[
    df_model["inchikey14"].isin(val_ids)
].reset_index(drop=True)

print("Total sampled spectra:", len(df_model))
print("Training spectra:", len(df_train))
print("Validation spectra:", len(df_val))

assert len(df_train) > 0
assert len(df_val) > 0

assert set(df_train["inchikey14"]).isdisjoint(
    set(df_val["inchikey14"])
)

print("Train/Validation separation passed!")

## 11. PyTorch Dataset and DataLoader

We prepare the processed MS/MS spectra for CNN training.

### Input Features
Each spectrum is converted into a 2200-dimensional
vector using 1 Da binning and square-root intensity
transformation.

### Prediction Targets
Each molecular structure is represented by a
1024-bit Morgan fingerprint generated from its
original SMILES using RDKit.

### Dataset Design
- Apply spectrum binning on demand
- Cache molecular fingerprints to reduce repeated work
- Keep training and validation datasets separate
- Exclude known ambiguous groups from initial training
- Preserve the complete validation subset

This prototype uses approximately 20,000 spectra
to verify the training pipeline before scaling up.

In [ ]:
from functools import lru_cache

import numpy as np
import torch
from rdkit import rdBase
from rdkit.Chem import rdFingerprintGenerator

print("RDKit version:", rdBase.rdkitVersion)

# Preprocessing configuration
MZ_MAX = 2200
BIN_WIDTH = 1.0

FP_SIZE = 1024
FP_RADIUS = 2

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS,
    fpSize=FP_SIZE,
    includeChirality=False
)


def spectrum_to_bins(mzs, intensities):
    """
    Convert variable-length MS/MS peaks into
    a fixed-length spectrum vector.
    """

    mzs = np.asarray(mzs, dtype=np.float64)
    intensities = np.asarray(intensities, dtype=np.float32)

    if len(mzs) != len(intensities):
        raise ValueError("Spectrum array lengths differ")

    n_bins = int(np.ceil(MZ_MAX / BIN_WIDTH))

    vector = np.zeros(n_bins, dtype=np.float32)

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs >= 0)
        & (mzs < MZ_MAX)
        & (intensities >= 0)
    )

    mzs = mzs[valid]
    intensities = intensities[valid]

    indices = np.floor(
        mzs / BIN_WIDTH
    ).astype(np.int64)

    values = np.sqrt(intensities)

    np.maximum.at(vector, indices, values)

    return vector


@lru_cache(maxsize=20000)
def smiles_to_fingerprint(smiles):
    """
    Convert SMILES to a 1024-bit Morgan fingerprint.
    """

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    fp = morgan_generator.GetFingerprint(mol)

    arr = np.zeros(FP_SIZE, dtype=np.uint8)

    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr

In [ ]:
from torch.utils.data import DataLoader, Dataset


class SpectrumFingerprintDataset(Dataset):

    def __init__(self, dataframe):

        self.records = dataframe.to_dict("records")

    def __len__(self):

        return len(self.records)

    def __getitem__(self, idx):

        row = self.records[idx]

        # Spectrum input
        spectrum = spectrum_to_bins(
            row["ms2_mzs"],
            row["ms2_normalized_intensities"]
        )

        # Molecular fingerprint target
        fingerprint = smiles_to_fingerprint(
            row["normalized_smiles"]
        )

        x = torch.from_numpy(
            spectrum
        ).unsqueeze(0)

        y = torch.from_numpy(
            fingerprint.astype(np.float32)
        )

        return x, y


train_dataset = SpectrumFingerprintDataset(df_train)
val_dataset = SpectrumFingerprintDataset(df_val)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created successfully!")

In [ ]:
X_batch, y_batch = next(iter(train_loader))

print("Input shape:", X_batch.shape)
print("Target shape:", y_batch.shape)

print("\nInput dtype:", X_batch.dtype)
print("Target dtype:", y_batch.dtype)

print("\nInput minimum:", X_batch.min().item())
print("Input maximum:", X_batch.max().item())

print("\nTarget unique values:", torch.unique(y_batch))

print(
    "Average active fingerprint bits:",
    y_batch.sum(dim=1).float().mean().item()
)

# Check validation batch
X_val, y_val = next(iter(val_loader))

print("\nValidation input shape:", X_val.shape)
print("Validation target shape:", y_val.shape)

# Assertions
assert X_batch.shape == (32, 1, 2200)
assert y_batch.shape == (32, 1024)

assert X_val.shape[1:] == (1, 2200)
assert y_val.shape[1:] == (1024,)

assert torch.isfinite(X_batch).all()
assert torch.isfinite(y_batch).all()

assert X_batch.min() >= 0
assert X_batch.max() <= 1

assert torch.all((y_batch == 0) | (y_batch == 1))

print("\nAll DataLoader tests passed!")

## 12. Save Prototype Datasets

The initial PyTorch DataLoader tests passed.

We save the sampled training and validation datasets
as Parquet files for reproducible CNN experiments.

Both CNN architectures will use the same data split,
input preprocessing, and fingerprint configuration
to ensure a fair comparison.

In [ ]:
from pathlib import Path

PROTOTYPE_DIR = SAVE_DIR / "prototype_data"
PROTOTYPE_DIR.mkdir(parents=True, exist_ok=True)

train_path = PROTOTYPE_DIR / "train_prototype.parquet"
val_path = PROTOTYPE_DIR / "val_prototype.parquet"

df_train.to_parquet(train_path, index=False)
df_val.to_parquet(val_path, index=False)

print("Saved training data:", train_path)
print("Saved validation data:", val_path)

print("\nTraining samples:", len(df_train))
print("Validation samples:", len(df_val))

# Verify saved files
assert train_path.exists()
assert val_path.exists()

print("\nPrototype datasets saved successfully!")